# Hyperparameter Tuning with Optuna — Olivetti Faces ANN

## Overview

A neural network has two kinds of "knobs":

- **Parameters** — the weights the network *learns* from data during training.
- **Hyperparameters** — settings *we* choose *before* training: number of layers, layer width, optimizer, learning rate, dropout, and so on.

The network can't learn its own hyperparameters, and a strong architecture with a bad learning rate loses to a simple one that's well tuned. **Optuna** searches for good hyperparameters automatically: we define a search space and a score to maximise (here, test accuracy), and it proposes combinations, learns from the results, and homes in on the best — more efficiently than grid or random search.

**Goal:** build an ANN that classifies 64×64 grayscale face images into 40 identities (the Olivetti Faces dataset), with its hyperparameters tuned by Optuna.

| Step | What happens |
|------|--------------|
| 1 | Imports, reproducibility, device (CPU/GPU) setup |
| 2 | Load and visualise the Olivetti Faces data |
| 3 | Preprocess: split, wrap in a `Dataset` |
| 4 | Define a **parameterized** network — **you write this** |
| 5 | Write the Optuna **objective** function — **you define the search space** |
| 6 | Create a **study** and run the search |
| 7 | Read off the best score and hyperparameters |

Run the cells top-to-bottom. Cells marked **TODO** are for you to complete — everything else is provided so you can focus on the two parts that matter most: the model architecture and the hyperparameter search space.

## Step 1 — Imports, reproducibility & device

Import PyTorch (model + training), scikit-learn (data), and matplotlib (plots). We fix random seeds so runs are repeatable, and detect a GPU (training uses it automatically if present).

**Note:** on Colab, enable a GPU via *Runtime → Change runtime type → GPU* — the device line will then print `cuda` and training will be much faster (though this dataset is small enough that CPU works fine too).

In [ ]:
import numpy as np
import random

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

In [ ]:
# Set random seeds so every run is reproducible.
# Optuna still explores different hyperparameters each trial, but the randomness
# *inside* training (weight initialisation, batch shuffling) stays fixed.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 2 — Load and explore the data

**Olivetti Faces**: 400 grayscale images (64×64) of 40 people (10 images each), taken at different times, lighting, expressions, and with/without glasses. `sklearn.datasets.fetch_olivetti_faces` downloads it automatically the first time it's called, and the pixel values already come scaled to **[0, 1]** — no need to divide by 255 here.

In [ ]:
olivetti = fetch_olivetti_faces()

X = olivetti.data      # shape (400, 4096) — flattened 64x64 images, already scaled to [0, 1]
y = olivetti.target    # shape (400,) — integer label 0-39, one per person
images = olivetti.images  # shape (400, 64, 64) — same data, kept 2D for visualisation

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of classes:", len(np.unique(y)))

In [ ]:
# A quick look at a handful of faces and their labels.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i], cmap='gray')
    ax.set_title(f"Person {y[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 3 — Preprocess

1. **Train / test split** — with only 10 images per person, we `stratify=y` so every person is represented in both the train and test sets.
2. **Scaling** — already done for us (`fetch_olivetti_faces` returns pixels in [0, 1]).
3. **Wrap in a `Dataset`** — so a `DataLoader` can feed the model mini-batches.

**Note:** the `DataLoader` itself is created later, *inside* the objective function — because `batch_size` is a hyperparameter Optuna will tune.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    # Convert to PyTorch tensors
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
# Sanity check: how many samples ended up in each split?
len(train_dataset), len(test_dataset)

## Step 4 — A parameterized network **(TODO)**

Rather than hard-coding one architecture, `MyNN` should be built from its constructor arguments: `num_hidden_layers` (depth), `neurons_per_layer` (width), and `dropout_rate` (regularisation) — so that Optuna can reshape the whole network just by changing numbers.

Each hidden block should be `Linear → BatchNorm1d → ReLU → Dropout`, repeated `num_hidden_layers` times, followed by one final `Linear` producing `output_dim` class scores.

Fill in the **TODO**s below.

In [ ]:
class MyNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__()

    layers = []

    # TODO: loop `num_hidden_layers` times. In each iteration, append (in order):
    #   nn.Linear(input_dim, neurons_per_layer)
    #   nn.BatchNorm1d(neurons_per_layer)
    #   nn.ReLU()
    #   nn.Dropout(dropout_rate)
    # then update input_dim = neurons_per_layer so the next block's input matches.


    # TODO: append the final nn.Linear(neurons_per_layer, output_dim) that produces
    # the class scores.


    self.model = None  # TODO: wrap `layers` in nn.Sequential(*layers)

  def forward(self, x):

    # TODO: pass x through self.model and return the result
    return None

## Step 5 — The Optuna objective function **(TODO: search space)**

Optuna calls this once per **trial**: given a set of hyperparameters, build the model, train it, and return the score to maximise (test accuracy). For each hyperparameter you call `trial.suggest_*`; Optuna picks the next value based on what it learned from previous trials.

The training loop, model construction, and evaluation are already written below — you only need to fill in the eight `trial.suggest_*` calls at the top. Use this as a starting point for your search space (feel free to reason about why, given only ~320 training images across 40 classes):

| Hyperparameter | Optuna call | Suggested range / choices |
|---|---|---|
| Hidden layers | `suggest_int` | 1 – 4 |
| Neurons / layer | `suggest_int(step=32)` | 64 – 256 |
| Epochs | `suggest_int(step=10)` | 10 – 50 |
| Learning rate | `suggest_float(log=True)` | 1e-5 – 1e-1 |
| Dropout | `suggest_float(step=0.1)` | 0.1 – 0.5 |
| Batch size | `suggest_categorical` | 8, 16, 32, 64 |
| Optimizer | `suggest_categorical` | Adam, SGD, RMSprop |
| Weight decay | `suggest_float(log=True)` | 1e-5 – 1e-3 |

**Why `log=True`?** Learning rate and weight decay span orders of magnitude — `0.001` vs `0.01` matters more than `0.05` vs `0.06` — so log-scale sampling spends the budget where it counts.

**Why smaller batch sizes than the Fashion-MNIST example?** The training split here has only ~320 images — a batch of 128 would barely give a handful of updates per epoch.

In [ ]:
# objective function: Optuna calls this once per trial.
# It receives a `trial`, suggests hyperparameters from it, trains a model,
# and returns the score to maximise (test accuracy).
def objective(trial):

  # TODO: ask the trial to suggest a value for each hyperparameter (the search space).
  # Use the table above as a guide. Replace each None with the matching trial.suggest_* call.
  num_hidden_layers = None   # TODO: trial.suggest_int("num_hidden_layers", ...)
  neurons_per_layer = None   # TODO: trial.suggest_int("neurons_per_layer", ..., step=...)
  epochs = None              # TODO: trial.suggest_int("epochs", ..., step=...)
  learning_rate = None       # TODO: trial.suggest_float("learning_rate", ..., log=True)
  dropout_rate = None        # TODO: trial.suggest_float("dropout_rate", ..., step=...)
  batch_size = None          # TODO: trial.suggest_categorical("batch_size", [...])
  optimizer_name = None      # TODO: trial.suggest_categorical("optimizer", [...])
  weight_decay = None        # TODO: trial.suggest_float("weight_decay", ..., log=True)

  # --- everything below this line is already written for you ---

  # DataLoaders are built here (not earlier) because batch_size is tuned.
  # pin_memory only helps when copying to a CUDA GPU, so enable it conditionally.
  pin = (device.type == 'cuda')
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=pin)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=pin)

  # model init
  input_dim = 4096   # 64 x 64 flattened pixels
  output_dim = 40    # 40 people

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

  # loss + optimizer selection
  # Each branch ASSIGNS `optimizer = ...` so the suggested optimizer,
  # learning_rate and weight_decay are actually used during training.

  criterion = nn.CrossEntropyLoss()

  if optimizer_name == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop
  model.train()
  for epoch in range(epochs):

    for batch_features, batch_labels in train_loader:

      # move data to the selected device (CPU/GPU)
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # calculate loss
      loss = criterion(outputs, batch_labels)

      # backward pass
      optimizer.zero_grad()
      loss.backward()

      # update weights
      optimizer.step()

  # evaluation on the held-out test set
  model.eval()
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in test_loader:

      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct / total

  # Optuna will try to maximise this returned value.
  return accuracy

## Step 6 — Create a study and run the search

A **study** manages the optimisation. We set `direction='maximize'` (higher accuracy is better; for a loss you'd use `'minimize'`). `study.optimize(objective, n_trials=...)` runs the objective that many times with hyperparameters chosen by Optuna's default **TPE** sampler, which focuses future trials on promising regions.

**Runtime:** each trial trains a full network, so more trials and a wider `epochs` range take longer. Start with a small `n_trials` (e.g. 10) to make sure everything runs end-to-end before scaling up.

In [ ]:
# Install Optuna if it isn't already available.
# (Colab usually needs this; skip it if Optuna is already installed locally.)
!pip install optuna

In [ ]:
import optuna

study = optuna.create_study(direction='maximize')

In [ ]:
study.optimize(objective, n_trials=10)

## Step 7 — Inspect the results

The study remembers its best trial: `study.best_value` (highest test accuracy) and `study.best_params` (the winning combination). These `best_params` are what you'd use to train a final model before deployment.

In [ ]:
study.best_value

In [ ]:
study.best_params

## Key takeaways

- **Hyperparameters are searched, not learned** — Optuna does this in a few lines.
- **The objective function is the contract:** take a `trial`, train a model, return a score. Everything else is ordinary PyTorch.
- **`trial.suggest_*` defines the search space**, and the TPE sampler beats grid/random for the same budget.
- With only ~320 training images and 40 classes, this is a much smaller, faster search than the Fashion-MNIST version — a good dataset for experimenting with `n_trials` and the search space itself.

**Next:** raise `n_trials`; visualise with `plot_optimization_history` / `plot_param_importances`; add a `MedianPruner`; or retrain a model on `study.best_params`.